# Project - AirLine AI Agent

In [64]:
import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio
import ollama

In [65]:
load_dotenv(override=True)

openai_api_key = os.getenv("OPENAI_API_KEY")
groq_api_key = os.getenv("GROQ_API_KEY")

groq_url = "https://api.groq.com/openai/v1"
ollama_url = "http://localhost:11434/v1"

openai = OpenAI(api_key=openai_api_key)
groq = OpenAI(api_key=groq_api_key, base_url=groq_url)
ollama = OpenAI(base_url=ollama_url, api_key="ollama")

MODEL = "openai/gpt-oss-120b"

In [66]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [67]:
def chat(message, history):
    chat_history = []

    for h in history:
        chat_history.append({"role": h['role'], "content": h['content']})

    messages = [{"role": "system", "content": system_message}] + chat_history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(
        model=MODEL,
        messages=messages
    )

    return response.choices[0].message.content




# Tools

In [68]:
ticket_prices = {
    "london": "$799",
    "paris": "$899",
    "tokyo": "$1400",
    "berlin": "$499"

}

In [69]:
def get_ticket_price(destination_city):
    print("Tool called for city :", destination_city)
    price = ticket_prices.get(destination_city.lower())
    return "The price for a ticket to " + destination_city + " is " + price

In [70]:
get_ticket_price("tokyo")

Tool called for city : tokyo


'The price for a ticket to tokyo is $1400'

In [71]:
get_price_tool = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to" ,
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}

In [72]:
# tools

tools = [
    {"type": "function", "function": get_price_tool}
]

In [73]:
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

# Flow
 Define Tools -> send LLMs -> Question -> Tool required -> send tool information -> Execute

In [74]:

def chat(message, history):
    chat_history = []

    for h in history:
        chat_history.append({"role": h['role'], "content": h['content']})

    messages = [{"role": "system", "content": system_message}] + chat_history + [{"role": "user", "content": message}]
    response = groq.chat.completions.create(model=MODEL,messages=messages,tools=tools)

    while response.choices[0].finish_reason == 'tool_calls':
        message = response.choices[0].message
        response = handle_tool_call(message)
        messages.append(message)
        messages.extend(response)
        response = groq.chat.completions.create(model=MODEL,messages=messages, tools=tools)

    return response.choices[0].message.content



In [75]:
def handle_tool_call(message):
    response =[]
    for tool_call in message.tool_calls:
        if tool_call.function.name == 'get_ticket_price':
            args = json.loads(tool_call.function.arguments)
            city_name = args.get("destination_city")
            price = get_ticket_price(city_name)
            response.append({
                "role": "tool",
                "content": price,
                "tool_call_id": tool_call.id
           })

    return response

In [ ]:
gradio.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7868
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Berlin


: 

# Database

In [98]:
import sqlite3

In [99]:
DB = "prices.db"

with sqlite3.connect(DB) as conn:
    cursor = conn.cursor()
    cursor.execute("CREATE TABLE IF NOT EXISTS ticket_prices (city TEXT PRIMARY KEY, price REAL)")
    conn.commit()

    

In [105]:
def set_ticket_price(city, price):
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute(
            'INSERT INTO ticket_prices (city, price) VALUES (?, ?) '
            'ON CONFLICT(city) DO UPDATE SET price = excluded.price',
            (city.lower(), price)
        )
        conn.commit()

In [106]:
ticket_prices = {"london": 799, "paris": 899, "tokyo": 1400,"berlin": 499}
for city, price in ticket_prices.items() :
     set_ticket_price(city, price)

In [108]:
def get_ticket_price(city) :
    print (f"DATABASE TOOL CALLED: Getting price for {city}", flush=True) 
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM ticket_prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [109]:
get_ticket_price("tokyo")

DATABASE TOOL CALLED: Getting price for tokyo


'Ticket price to tokyo is $1400.0'